In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:100% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:20pt;}
div.text_cell_render.rendered_html{font-size:18pt;}
div.text_cell_render.rendered_html{font-size:15pt;}
div.output {font-size:18pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:18pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:18pt;padding:5px;}
table.dataframe{font-size:18px;}
</style>
"""))

**<font size="6" color="red">ch3. 연관분석</font>**
- pip install apyori

# 1. 연관분석 개요
- 데이터들 사이에 자주 발생하는 속성을 찾고, 그 속성들 사이에 연관성이 어느 정도 있는지 분석
- 활용분야 : 이벤트미리감지(사기적발..), 신상품카테고리 구성

[조건:left-hand side:오렌지주소]->[결과:right-hand side:와인]
```
- 연과분석과 관련된 지표
1. 지지도(support) : 얼마나 자주 함께 나타나는지
    (lhs, rhs)의 항목수/전체항목수 = 0.2
    
2. 신뢰도(confidence) : 조건이 오면 결과가 얼마나 자주 나타나는지
    (lhs->rhs)의 항목수/lhs의 항목수 = 1/2 = 0.5
    
3. 향상도(lift) : 우연히 발생한 규칙은 아닌지 확인
    lhs->rhs의 지지도 / (lhs의 지지도*rhs의 지지도)
    => 0.2 / (0.4*0.6) = 0.2/0.24 = 0.833
    향상도<1 : 기대가 낮다
    향상도>1 : 기대가 높다
```

# 2. 연관분석 구현

In [1]:
import csv
with open('data/cf_basket.csv', 'r', encoding='utf-8') as f:
    csvdata = csv.reader(f)
    # print(list(csvdata))
    transaction = list(csvdata)
transaction

[['소주', '콜라', '와인'],
 ['소주', '오렌지주스', '콜라'],
 ['맥주', '콜라', '와인'],
 ['소주', '콜라', '맥주'],
 ['오렌지주스', '와인']]

In [2]:
from apyori import apriori
rules = apriori(transaction, # 2차원 데이터
               min_support=0.15,
               min_confidence=0.1)
rules = list(rules)
len(rules)

18

In [3]:
rules[10]

RelationRecord(items=frozenset({'소주', '콜라'}), support=0.6, ordered_statistics=[OrderedStatistic(items_base=frozenset(), items_add=frozenset({'소주', '콜라'}), confidence=0.6, lift=1.0), OrderedStatistic(items_base=frozenset({'소주'}), items_add=frozenset({'콜라'}), confidence=1.0, lift=1.25), OrderedStatistic(items_base=frozenset({'콜라'}), items_add=frozenset({'소주'}), confidence=0.7499999999999999, lift=1.2499999999999998)])

In [4]:
rule = rules[10]
support = rule[1]
order_st = rule[2]
for item in order_st:
    lhs = item[0]
    rhs = item[1]
    confidence = item[2]
    lift = item[3]
    if lift > 1:
        print("{}=>{}\t {}\t {}\t {}".format(lhs, rhs, support, 
                                             round(confidence,2), 
                                             round(lift,2)))

frozenset({'소주'})=>frozenset({'콜라'})	 0.6	 1.0	 1.25
frozenset({'콜라'})=>frozenset({'소주'})	 0.6	 0.75	 1.25


In [5]:
for rule in rules:
    support = rule[1]
    order_st = rule[2]
    for item in order_st:
        lhs = item[0]
        rhs = item[1]
        confidence = item[2]
        lift = item[3]
        if lift > 1:
            print("{}=>{}\t {}\t {}\t {}".format(lhs, rhs, support, 
                                                 round(confidence,2), 
                                                 round(lift,2)))

frozenset({'맥주'})=>frozenset({'콜라'})	 0.4	 1.0	 1.25
frozenset({'콜라'})=>frozenset({'맥주'})	 0.4	 0.5	 1.25
frozenset({'소주'})=>frozenset({'콜라'})	 0.6	 1.0	 1.25
frozenset({'콜라'})=>frozenset({'소주'})	 0.6	 0.75	 1.25
frozenset({'콜라'})=>frozenset({'맥주', '소주'})	 0.2	 0.25	 1.25
frozenset({'맥주', '소주'})=>frozenset({'콜라'})	 0.2	 1.0	 1.25
frozenset({'맥주'})=>frozenset({'콜라', '와인'})	 0.2	 0.5	 1.25
frozenset({'콜라'})=>frozenset({'맥주', '와인'})	 0.2	 0.25	 1.25
frozenset({'맥주', '와인'})=>frozenset({'콜라'})	 0.2	 1.0	 1.25
frozenset({'콜라', '와인'})=>frozenset({'맥주'})	 0.2	 0.5	 1.25
frozenset({'소주'})=>frozenset({'콜라', '오렌지주스'})	 0.2	 0.33	 1.67
frozenset({'콜라'})=>frozenset({'소주', '오렌지주스'})	 0.2	 0.25	 1.25
frozenset({'소주', '오렌지주스'})=>frozenset({'콜라'})	 0.2	 1.0	 1.25
frozenset({'콜라', '오렌지주스'})=>frozenset({'소주'})	 0.2	 1.0	 1.67
frozenset({'콜라'})=>frozenset({'소주', '와인'})	 0.2	 0.25	 1.25
frozenset({'소주', '와인'})=>frozenset({'콜라'})	 0.2	 1.0	 1.25


In [ ]:
import pandas as pd
rules_df = pd.DataFrame(None, columns=['lhs', 'rhs', '지지도', '신뢰도', '향상도'])
# rules_df.loc[0] = ['와인', '오렌지', 0.15, 0.5, 1.1] 식으로 for문내에서 데이터추가
idx = 0


,lhs,rhs,지지도,신뢰도,향상도
0,와인,오렌지,0.15,0.5,1.1
